> [! INFO] 서울시 공공자전거 따릉이 실시간 대여정보
>
> * 📊 [데이터 출처 (서울 열린데이터 광장)](https://data.seoul.go.kr/dataList/OA-15493/A/1/datasetView.do)

In [ ]:
# autoreload 이라는 모듈을 자동으로 다시 불러오는 IPython의 확장 기능을 사용하겠다
%load_ext autoreload
# 코드가 실행되기 전, 모든 모듈을 무조건 자동으로 새로고침합니다.
%autoreload 2

In [6]:
import sys
import tomllib
import io
import boto3
import polars as pl
from datetime import datetime
from pathlib import Path
from sqlalchemy import create_engine, text

In [7]:
today = datetime.now().strftime("%Y-%m-%d")
ROOT_DIR = Path.cwd()  # Currnet Dir(notebooks)

print(ROOT_DIR)
toml_path = ROOT_DIR / "config" / "dev.toml"

if str(ROOT_DIR) not in sys.path:    # 다음셀 부터 프로젝트 최상위 폴더"seoul_public_bike" 위치를 강제로 등록
    sys.path.append(str(ROOT_DIR))

from utils.openapi_client import OpenApiClient

D:\workspace\seoul_public_bike\src\notebook\seoul-public-bike


In [8]:
with open(toml_path, "rb") as f:
    config = tomllib.load(f)

service_name = config["seoul_api_dim"]["service_name"]  # "bikeList"
api_key = config["seoul_api_global"]["api_key"]

In [9]:
open_api_client:OpenApiClient = OpenApiClient(api_key)
first_dim = open_api_client.fetch_dimension(service_name)

[Dimension] bikeList 수집 시작
ℹ️ [디버그 5] 마지막 페이지에 도달했습니다. (가져온 행: 736개)
✅ 디멘전 수집 완료: 총 2736개 행



In [8]:
# if not first_dim:
#     print(f"금일 [{service_name}] 공공 API 데이터가 비어있습니다. 적재를 패스합니다.")
# else :
# # 🚀 데이터가 있을 때만 훑고 디스크에 굽는 메인 로직 가동
#     dim_dir = ROOT_DIR / "datalake" / "seoul-public-bike" / "dim" / service_name / f"snapshot_dt={today}"
#     dim_dir.mkdir(parents=True, exist_ok=True)
#     dim_path = dim_dir / "bikeList.parquet"
#
#     df = pl.DataFrame(first_dim)
#     df.write_parquet(dim_path) # saved_df = 지우기
#
#     print("📋 데이터 스캔 성공! 구조:", df.shape)
#     print(df.schema) # 🌟 df 자체의 스키마를 출력해야 에러가 안 납니다!

[Dimension] bikeList 수집 시작
✅ 디멘전 수집 완료: 총 0개 행

금일 [bikeList] 공공 API 데이터가 비어있습니다. 적재를 패스합니다.


In [9]:
# import pyarrow.parquet as pq
#
# # 파케트 파일 메타데이터 읽기
# metadata = pq.read_metadata(dim_path)
# print(f"전체 행 개수: {metadata.num_rows}개")
# print(f"내부적으로 쪼개진 방(Row Group) 개수: {metadata.num_row_groups}개")
#
# # 첫 번째 방에 몇 개의 행이 들어있는지 확인
# if metadata.num_row_groups > 0:
#     print(f"첫 번째 로우 그룹의 행 개수: {metadata.row_group(0).num_rows}개")


전체 행 개수: 0개
내부적으로 쪼개진 방(Row Group) 개수: 0개


#### S3

In [12]:
aws_config = config["aws"]
BUCKET_NAME = aws_config["bucket_name"]
ACCESS_KEY = aws_config["access_key_id"]
SECRET_KEY = aws_config["secret_access_key"]
REGION = aws_config["region"]

df = pl.DataFrame(first_dim)

# ==========================================
# 3. 📦 메모리에 안전하게 파케트 박스 포장하기
# ==========================================
# 로컬 SSD를 거치지 않고, 컴퓨터 메모리 상에 가상의 파일 공간을 만들어 파케트로 굽습니다.
buffer = io.BytesIO()
df.write_parquet(buffer, compression="snappy")
buffer.seek(0)  # 포장된 파일의 처음 위치로 커서를 돌려놓습니다.

s3_client = boto3.client(
    "s3",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION,
)

s3_key = (
    f"seoul-public-bike/dim/{service_name}/snapshot_dt={today}/{service_name}.parquet"
)

try:
    print(f" [boto3] AWS 공식 라인을 타고 S3로 파케트 파일 전송 시작...")

    s3_client.put_object(
        Bucket=BUCKET_NAME,
        Key=s3_key,  # 저장될 가상 폴더 구조와 파일명
        Body=buffer.getvalue(),  # 메모리에 포장해둔 파케트 데이터 알맹이
    )

    print(" [대성공] S3에 파케트 파일이 정상 적재되었습니다!")
    print(f" 확인 경로: s3://{BUCKET_NAME}/{s3_key}")

except Exception as e:
    print(f"전송 실패 (열쇠 철자나 버킷명을 다시 확인해보세요): {e}")

 [boto3] AWS 공식 라인을 타고 S3로 파케트 파일 전송 시작...
 [대성공] S3에 파케트 파일이 정상 적재되었습니다!
 확인 경로: s3://my-fast-test-datalake-bucket-637423225965-ap-northeast-2-an/seoul-public-bike/dim/bikeList/snapshot_dt=2026-07-21/bikeList.parquet


In [13]:
s3_path = f"s3://{BUCKET_NAME}/{s3_key}"

storage_options = {
    "aws_access_key_id": ACCESS_KEY,
    "aws_secret_access_key": SECRET_KEY,
    "aws_region": REGION,
}

lazy_df = pl.scan_parquet(s3_path, storage_options=storage_options)
as_cleaned_df = lazy_df.select(pl.all().name.to_lowercase())
final_df = as_cleaned_df.collect(engine="streaming")
print(final_df)

shape: (2_736, 7)
┌────────────┬────────────────┬───────────────┬────────┬───────────────┬───────────────┬───────────┐
│ racktotcnt ┆ stationname    ┆ parkingbiketo ┆ shared ┆ stationlatitu ┆ stationlongit ┆ stationid │
│ ---        ┆ ---            ┆ tcnt          ┆ ---    ┆ de            ┆ ude           ┆ ---       │
│ str        ┆ str            ┆ ---           ┆ str    ┆ ---           ┆ ---           ┆ str       │
│            ┆                ┆ str           ┆        ┆ str           ┆ str           ┆           │
╞════════════╪════════════════╪═══════════════╪════════╪═══════════════╪═══════════════╪═══════════╡
│ 15         ┆ 102. 망원역    ┆ 13            ┆ 87     ┆ 37.55564880   ┆ 126.91062927  ┆ ST-4      │
│            ┆ 1번출구 앞     ┆               ┆        ┆               ┆               ┆           │
│ 14         ┆ 103. 망원역    ┆ 16            ┆ 114    ┆ 37.55495071   ┆ 126.91083527  ┆ ST-5      │
│            ┆ 2번출구 앞     ┆               ┆        ┆               ┆               

In [11]:
# # 1. read_parquet 대신 scan_parquet을 씁니다. (메모리에 올리지 않고 스캔만 함)
# lazy_df = pl.scan_parquet(dim_path)
# as_cleaned_df = lazy_df.select(pl.all().name.to_lowercase())
# final_df = as_cleaned_df.collect(engine="streaming")
# print(final_df)

shape: (0, 0)
┌┐
╞╡
└┘


In [14]:
from utils.db_client import BaseDBClient, PostgresDBClient
db_config = config.get("database", {})
db_clidnt:BaseDBClient = PostgresDBClient(**db_config)

df_url = db_clidnt.get_connection_uri()

In [25]:
engine = create_engine(df_url)
table_with_schema = "ods.dim_bike_station"

ods = f"""
CREATE SCHEMA IF NOT EXISTS ods;

DROP TABLE IF EXISTS {table_with_schema};

CREATE TABLE {table_with_schema} (
    racktotcnt TEXT,
    stationname TEXT,
    parkingbiketotcnt TEXT,
    shared TEXT,
    stationlatitude TEXT,
    stationlongitude TEXT,
    stationid TEXT
);
"""
if final_df.width == 0 or final_df.is_empty():
    print("[패스] 금일 대여소 마스터(dim) 데이터가 비어있어 DB 적재를 안전하게 건너뜁니다.")

else:
    print(f"[시작] 총 {len(final_df)}개의 마스터 데이터를 ods 레이어에 적재합니다.")

    with engine.connect() as conn:
        conn.execute(text(ods))
        conn.commit()

    final_df.write_database(
        table_name=table_with_schema,
        connection=df_url,
        if_table_exists="append",
        engine="adbc"
    )
print("✅ ods.dim_bike_station 적재 완벽 성공!")

[시작] 총 2736개의 마스터 데이터를 ods 레이어에 적재합니다.
✅ ods.dim_bike_station 적재 완벽 성공!


In [32]:
dw_source_table = "ods.dim_bike_station"
dw_target_table = "dw.dim_bike_station"

dw_sql_template = (ROOT_DIR / "dw/dim_bike_station.sql").read_text(encoding="utf-8")

execute_dw = dw_sql_template.format(
    target_table=dw_target_table,
    source_table=dw_source_table
)

# 💡 DB 커넥션을 한 번만 열어서 내부에서 모든 작업을 연속 처리하네!
with engine.connect() as conn:
    # 1. DW 변환 및 Soft Delete SQL 실행 및 커밋
    conn.execute(text(execute_dw))
    conn.commit()

    # 2. 열려있는 커넥션 통로에서 바로 활성 대여소 수 조회!
    total_active_count = conn.scalar(
        text(f"SELECT COUNT(*) FROM {dw_target_table} WHERE is_deleted = FALSE")
    )

print(f" 현재 DW 내 전체 활성 대여소 수: {total_active_count}개")

 현재 DW 내 전체 활성 대여소 수: 2736개


In [16]:
# -----------------------------------------------------------------------
# 실행 2: DW -> DM (다음에 복붙해서 만들 녀석)
# -----------------------------------------------------------------------

dm_source_table =  "dw.dim_bike_station" # 💡 "ods.dim_bike_station"이 그대로 출발지가 됩니다!
dm_target_table = "dm.dim_bike_station"

dw_sql_template = (ROOT_DIR / "src/dw/dim_bike_station.sql").read_text(encoding="utf-8")

execute_dw = dw_sql_template.format(
    target_table=dw_target_table,
    source_table=dw_source_table  # 💡 ODS의 타겟 변수가 여기 소스로 쏙 들어감!
)

with engine.connect() as conn:
    conn.execute(text(execute_dw))
    conn.commit()
print("✅ 2단계: DW 이력 적재 완료!")

✅ 2단계: DW 이력 적재 완료!
